# Linear Algebra for Deep Learning — A PyTorch Tutorial

This notebook is a hands-on companion to the linear algebra concepts that show up
constantly in deep learning, taught **through PyTorch code** rather than pure math.

**Structure:** 7 modules + a capstone project. Each module has:
- **Math Basics** — the concept, explained briefly with intuition
- **PyTorch Implementation** — the code that puts it into practice
- **Exercise** — a "predict before you run" task for you to try
- **Solution** — check your work after attempting the exercise

## Setup: Install Python, pip, and Required Packages

If you do not already have Python 3 and `pip`, install them first for your operating system:

### Windows
1. Install Python 3 from <https://www.python.org/downloads/windows/>.
2. During installation, check **Add Python to PATH**.
3. Open PowerShell and verify:
   ```powershell
   python --version
   pip --version
   ```
4. Install the packages:
   ```powershell
   pip install torch numpy matplotlib scikit-learn
   ```

### macOS
Using Homebrew:
```bash
brew install python
python3 --version
python3 -m pip --version
python3 -m pip install torch numpy matplotlib scikit-learn
```

If you do not have Homebrew, install Python 3 from <https://www.python.org/downloads/macos/>.

### Ubuntu / Debian Linux
```bash
sudo apt update
sudo apt install python3 python3-pip
python3 --version
python3 -m pip --version
python3 -m pip install torch numpy matplotlib scikit-learn
```

### Fedora Linux
```bash
sudo dnf install python3 python3-pip
python3 --version
python3 -m pip --version
python3 -m pip install torch numpy matplotlib scikit-learn
```

**Prerequisites:** Python basics, Python 3, `pip`, and the packages `torch`, `numpy`, `matplotlib`, and `scikit-learn`.

## Directory

1. **Module 0 — Tensors as the Universal Object**: tensor ranks, shapes, and basic tensor creation.
2. **Module 1 — Vectors: Operations, Norms, Similarity, Outer and Cross Products**: vector arithmetic, dot products, norms, cosine similarity, outer products, and cross products.
3. **Module 2 — Matrices: Multiplication, Transpose, Trace, Determinant, Orthogonality**: matrices as transformations, matrix multiplication, trace, determinants, and orthogonal matrices.
4. **Module 3 — Vectorization, Broadcasting, Reshaping, and Batched Operations**: replacing loops with tensor operations, broadcasting rules, reshaping, and batch matrix multiplication.
5. **Module 4 — Linear Transformations & `nn.Linear` Demystified**: affine layers, PyTorch's row-major batch layout, and manual `nn.Linear` calculations.
6. **Module 5 — Rank, Eigenvalues, SVD, PCA, and Low-Rank Approximation**: rank, eigenvectors, singular value decomposition, PCA, and matrix compression.
7. **Module 6 — Gradients, Jacobians, Autograd, and Positive Definiteness**: gradients, Jacobians, autograd, Hessians, and positive definite matrices.
8. **Capstone — A Tiny Neural Network Built From Raw Tensors**: a small neural network trained from scratch using raw tensors and autograd.
9. **Solutions**: collected answers for all exercises.

Let's get started.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
print("Torch version:", torch.__version__)

---
## Module 0 — Tensors as the Universal Object

### Math Basics

In deep learning, everything is a **tensor** — a generalization of scalars, vectors,
and matrices to any number of dimensions ("rank" or "order"):

| Object | Rank | Example |
|---|---|---|
| Scalar | 0 | a single number, `5` |
| Vector | 1 | `[1, 2, 3]` |
| Matrix | 2 | a 2D grid of numbers |
| Tensor | 3+ | e.g. a batch of images: `(batch, channels, height, width)` |

The **shape** of a tensor tells you the size along each axis (dimension). Getting
comfortable reading and predicting shapes is arguably the single most useful skill
for working with PyTorch.

### PyTorch Implementation

In [ ]:
# Scalar (rank 0)
scalar = torch.tensor(5.0)
print("scalar:", scalar, "| shape:", scalar.shape, "| ndim:", scalar.ndim)

# Vector (rank 1)
vector = torch.tensor([1.0, 2.0, 3.0])
print("vector:", vector, "| shape:", vector.shape, "| ndim:", vector.ndim)

# Matrix (rank 2)
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("matrix:\n", matrix, "\n| shape:", matrix.shape, "| ndim:", matrix.ndim)

# 3D tensor (rank 3) -- e.g. a tiny "batch" of 2x2 images
tensor3d = torch.randn(4, 2, 2)  # (batch=4, height=2, width=2)
print("3D tensor shape:", tensor3d.shape, "| ndim:", tensor3d.ndim)

### Exercise 0

Before running the code cell below, **write down on paper** what shape you expect
for each of these:

1. `torch.zeros(3, 5)` → ?
2. `torch.ones(2, 3, 4)` → ?
3. `torch.tensor([1, 2, 3]).unsqueeze(0)` → ?
4. `torch.tensor([[1, 2, 3]]).squeeze(0)` → ?

Then run the cell and check yourself.

In [ ]:
# Your turn: predict the shapes above, then uncomment and run to check
# print(torch.zeros(3, 5).shape)
# print(torch.ones(2, 3, 4).shape)
# print(torch.tensor([1, 2, 3]).unsqueeze(0).shape)
# print(torch.tensor([[1, 2, 3]]).squeeze(0).shape)

---
## Module 1 — Vectors: Operations, Norms, Similarity, Outer and Cross Products

### Math Basics

**Vector addition & scalar multiplication.** Geometrically, addition is
"tip-to-tail": place the second vector's tail at the first vector's tip, and the
sum is the arrow from the very start to the very end. Scalar multiplication
stretches or shrinks a vector, and flips its direction entirely if the scalar is
negative.

**Dot product — two equivalent definitions.**

$$
\mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i \quad \text{(algebraic)} \qquad = \quad \|\mathbf{a}\|\,\|\mathbf{b}\|\cos(\theta) \quad \text{(geometric)}
$$

where $\theta$ is the angle between the two vectors. These are the *same number*
computed two different ways — that equivalence is worth sitting with, because the
geometric form is what makes the dot product useful:

- **Positive** dot product → angle < 90° (vectors roughly point the same way)
- **Zero** → exactly orthogonal (90°)
- **Negative** → angle > 90° (vectors roughly oppose each other)

For **unit vectors** specifically (`‖a‖ = ‖b‖ = 1`), the formula collapses to
$\mathbf{\hat a} \cdot \mathbf{\hat b} = \cos(\theta)$ — the dot product *is* the
cosine of the angle, with no rescaling needed. This is the identity we'll
visualize below.

**Outer product.** $\mathbf{a} \otimes \mathbf{b} = \mathbf{a}\mathbf{b}^\top$
contrasts with the dot product in the cleanest possible way: dot product
combines two vectors into a **scalar**; outer product combines them into a
**matrix**. Geometrically, that matrix always has **rank 1** — applying it to any
vector $\mathbf{x}$ first projects $\mathbf{x}$ onto $\mathbf{b}$ (via the scalar
$\mathbf{b}^\top\mathbf{x}$), then rescales $\mathbf{a}$ by that amount:

$$
(\mathbf{a}\mathbf{b}^\top)\mathbf{x} = \mathbf{a}(\mathbf{b}^\top\mathbf{x})
$$

No matter what $\mathbf{x}$ is, the output always lands somewhere on the single
line spanned by $\mathbf{a}$. (Keep this picture in mind — the exact same formula
reappears in Module 6 as the gradient of a linear layer's weights.)

**Cross product.** In 3D, $\mathbf{a} \times \mathbf{b}$ produces a new vector
perpendicular to both inputs. Its direction follows the right-hand rule, and its
magnitude is the area of the parallelogram spanned by the two vectors:

$$
\|\mathbf{a} \times \mathbf{b}\| = \|\mathbf{a}\|\,\|\mathbf{b}\|\sin(\theta)
$$

That makes the cross product the standard tool for mesh face normals and for
constructing the missing axis of a right-handed coordinate frame.

**Norms — each has a distinct geometric meaning:**
- **L2** (Euclidean): straight-line distance from the origin — the Pythagorean theorem generalized to $n$ dimensions
- **L1** (Manhattan): distance if you can only move along grid axes, like walking city blocks
- **L∞**: the size of the single largest coordinate, ignoring everything else

The clearest way to *see* the difference is the **unit ball** — the set of all
vectors with norm exactly 1. Under each norm, that set is a different shape: a
circle (L2), a diamond (L1), a square (L∞). This shape difference is *why* L1
regularization tends to produce sparse weights (the diamond's corners sit exactly
on the coordinate axes, so the optimum often lands there, zeroing out some
weights) while L2 regularization shrinks weights smoothly without favoring
sparsity (the circle has no corners to land on).

**Cosine similarity:** $\cos(\theta) = \dfrac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$
— the dot product normalized by both magnitudes, so it always equals $\cos(\theta)$
regardless of vector length. That's why it's preferred over a raw dot product for
comparing embeddings that may have very different magnitudes.

### PyTorch Implementation

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Dot product -> scalar
dot = torch.dot(a, b)
print("dot product:", dot.item())

# Outer product -> matrix
outer = torch.outer(a, b)
print("outer product:\n", outer, "| shape:", outer.shape)

# Cross product -> vector perpendicular to both inputs (3D only)
cross = torch.cross(a, b, dim=0)
print("cross product:", cross)
print("dot(cross, a):", torch.dot(cross, a).item(), "| dot(cross, b):", torch.dot(cross, b).item())

# Norms
print("L1 norm:", torch.norm(a, p=1).item())
print("L2 norm:", torch.norm(a, p=2).item())
print("Linf norm:", torch.norm(a, p=float('inf')).item())

# Cosine similarity
import torch.nn.functional as F
cos_sim = F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0))
print("cosine similarity:", cos_sim.item())

### Geometric Showcase — Visualizing the Dot Product as an Angle

Let's make the identity $\hat{\mathbf{a}} \cdot \hat{\mathbf{b}} = \cos(\theta)$
concrete: plot two vectors, measure the angle between them two different ways
(via `arccos` of the dot product, and directly from their coordinates), and
confirm they match.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_vector_angle(v1, v2, ax, title):
    v1n, v2n = v1.numpy(), v2.numpy()
    ax.quiver(0, 0, v1n[0], v1n[1], angles='xy', scale_units='xy', scale=1,
               color='steelblue', width=0.012, label=r'$\mathbf{a}$')
    ax.quiver(0, 0, v2n[0], v2n[1], angles='xy', scale_units='xy', scale=1,
               color='darkorange', width=0.012, label=r'$\mathbf{b}$')
    lim = max(np.abs(v1n).max(), np.abs(v2n).max()) * 1.4
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal'); ax.legend(loc='upper left'); ax.set_title(title)

# Three example pairs: acute, orthogonal, obtuse angle
pairs = [
    (torch.tensor([2.0, 1.0]), torch.tensor([1.0, 2.0])),   # acute
    (torch.tensor([2.0, 0.0]), torch.tensor([0.0, 2.0])),   # orthogonal
    (torch.tensor([2.0, 1.0]), torch.tensor([-1.0, 2.0])),  # obtuse
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for (v1, v2), ax in zip(pairs, axes):
    cos_sim = torch.dot(v1, v2) / (torch.norm(v1) * torch.norm(v2))
    angle_from_dot = torch.acos(torch.clamp(cos_sim, -1.0, 1.0)) * 180 / torch.pi

    # Cross-check via coordinates directly (atan2 of each vector, take the difference)
    angle1 = torch.atan2(v1[1], v1[0])
    angle2 = torch.atan2(v2[1], v2[0])
    angle_from_coords = torch.abs(angle1 - angle2) * 180 / torch.pi
    if angle_from_coords > 180:
        angle_from_coords = 360 - angle_from_coords

    plot_vector_angle(v1, v2, ax,
                        f"cos(θ)={cos_sim.item():.2f}\n"
                        f"θ from dot product: {angle_from_dot.item():.1f}°\n"
                        f"θ from coordinates: {angle_from_coords.item():.1f}°")

plt.tight_layout()
plt.show()
print("The two angle computations agree in every case -- this IS what the dot product measures.")

### Geometric Showcase — Cross Products for Mesh Normals and Right-Handed Axes

The outer product $\mathbf{a}\mathbf{b}^\top$ makes a matrix, but the geometry
used for triangle normals is the **cross product**. Given one triangle in a mesh
with vertices $\mathbf{p}_0, \mathbf{p}_1, \mathbf{p}_2$, build two edge vectors:

$$
\mathbf{e}_1 = \mathbf{p}_1 - \mathbf{p}_0, \qquad
\mathbf{e}_2 = \mathbf{p}_2 - \mathbf{p}_0
$$

Their cross product points perpendicular to the triangle:

$$
\mathbf{n} = \frac{\mathbf{e}_1 \times \mathbf{e}_2}{\|\mathbf{e}_1 \times \mathbf{e}_2\|}
$$

This is exactly how face normals are computed for meshes such as the Stanford
Bunny: do the same calculation independently for every triangular face. The
vertex order matters. If you swap $\mathbf{p}_1$ and $\mathbf{p}_2$, the normal
flips direction because $\mathbf{e}_2 \times \mathbf{e}_1 = -(\mathbf{e}_1 \times \mathbf{e}_2)$.

The same rule gives the missing axis of a right-handed coordinate frame. If
$\hat{\mathbf{x}}$ points right and $\hat{\mathbf{y}}$ points up, then

$$
\hat{\mathbf{z}} = \hat{\mathbf{x}} \times \hat{\mathbf{y}}
$$

Reverse the order and you get the opposite direction, which is the common source
of inside-out mesh normals and left-handed/right-handed coordinate bugs.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def unit(v):
    return v / torch.linalg.norm(v)

def face_normal(vertices, face):
    p0, p1, p2 = vertices[face]
    e1 = p1 - p0
    e2 = p2 - p0
    return unit(torch.cross(e1, e2, dim=0))

# A tiny triangular mesh patch. A Stanford Bunny mesh uses the same per-face math,
# just with thousands of vertices and triangular faces instead of five faces.
vertices = torch.tensor([
    [-1.0, -1.0, 0.0],
    [ 1.0, -1.0, 0.0],
    [ 1.0,  1.0, 0.0],
    [-1.0,  1.0, 0.0],
    [ 0.0,  0.0, 1.4],
])
faces = torch.tensor([
    [0, 1, 4],
    [1, 2, 4],
    [2, 3, 4],
    [3, 0, 4],
    [0, 3, 2],
    [0, 2, 1],
])

normals = torch.stack([face_normal(vertices, face) for face in faces])
highlight_face = faces[0]
p0, p1, p2 = vertices[highlight_face]
e1 = p1 - p0
e2 = p2 - p0
n = face_normal(vertices, highlight_face)
n_flipped = face_normal(vertices, torch.tensor([0, 4, 1]))

# Right-hand convention: if x and y are unit, orthogonal axes, z = x cross y.
x_axis = torch.tensor([1.0, 0.0, 0.0])
y_axis = torch.tensor([0.0, 1.0, 0.0])
z_axis = torch.cross(x_axis, y_axis, dim=0)

fig = plt.figure(figsize=(13, 5.5))
ax_mesh = fig.add_subplot(1, 2, 1, projection='3d')
ax_axes = fig.add_subplot(1, 2, 2, projection='3d')

triangles = [[vertices[idx].numpy() for idx in face] for face in faces]
mesh = Poly3DCollection(triangles, alpha=0.35, facecolor='lightsteelblue', edgecolor='gray')
ax_mesh.add_collection3d(mesh)
highlight = Poly3DCollection([[p0.numpy(), p1.numpy(), p2.numpy()]], alpha=0.7,
                             facecolor='darkorange', edgecolor='black')
ax_mesh.add_collection3d(highlight)

for face, normal in zip(faces, normals):
    center = vertices[face].mean(dim=0)
    ax_mesh.quiver(*center.numpy(), *(normal * 0.35).numpy(), color='seagreen', linewidth=1.8)

center = vertices[highlight_face].mean(dim=0)
ax_mesh.quiver(*p0.numpy(), *e1.numpy(), color='steelblue', linewidth=2.5, label=r'$e_1=p_1-p_0$')
ax_mesh.quiver(*p0.numpy(), *e2.numpy(), color='purple', linewidth=2.5, label=r'$e_2=p_2-p_0$')
ax_mesh.quiver(*center.numpy(), *(n * 0.7).numpy(), color='crimson', linewidth=3,
               label=r'$n=normalize(e_1 \times e_2)$')
ax_mesh.set_title('One triangle normal inside a mesh')
ax_mesh.legend(loc='upper left', fontsize=8)
ax_mesh.set_box_aspect((1, 1, 0.9))
ax_mesh.set_xlim(-1.4, 1.4); ax_mesh.set_ylim(-1.4, 1.4); ax_mesh.set_zlim(-0.4, 1.8)
ax_mesh.set_xlabel('x'); ax_mesh.set_ylabel('y'); ax_mesh.set_zlabel('z')
ax_mesh.view_init(elev=24, azim=-55)

origin = torch.zeros(3)
for axis, color, label in [
    (x_axis, 'steelblue', r'$\hat{x}$'),
    (y_axis, 'darkorange', r'$\hat{y}$'),
    (z_axis, 'seagreen', r'$\hat{z}=\hat{x}\times\hat{y}$'),
]:
    ax_axes.quiver(*origin.numpy(), *axis.numpy(), color=color, linewidth=3, arrow_length_ratio=0.12)
    ax_axes.text(*(axis * 1.12).numpy(), label, color=color, fontsize=12)

ax_axes.set_title('Right-hand convention: x cross y gives z')
ax_axes.set_xlim(0, 1.25); ax_axes.set_ylim(0, 1.25); ax_axes.set_zlim(0, 1.25)
ax_axes.set_xlabel('x'); ax_axes.set_ylabel('y'); ax_axes.set_zlabel('z')
ax_axes.set_box_aspect((1, 1, 1))
ax_axes.view_init(elev=22, azim=-45)

plt.tight_layout()
plt.show()

print('highlighted triangle vertices:')
print('p0 =', p0.numpy(), 'p1 =', p1.numpy(), 'p2 =', p2.numpy())
print('\ne1 = p1 - p0:', e1.numpy())
print('e2 = p2 - p0:', e2.numpy())
print('normal = normalize(cross(e1, e2)):', n.numpy().round(4))
print('same triangle with reversed winding:', n_flipped.numpy().round(4))
print('dot(normal, reversed normal):', torch.dot(n, n_flipped).item(), '-> -1 means exactly opposite')

print('\nright-handed frame:')
print('x cross y =', z_axis.numpy(), 'so +z completes the frame')
print('y cross x =', torch.cross(y_axis, x_axis, dim=0).numpy(), 'which points the opposite way')

### Geometric Showcase — Unit Balls of L1, L2, L∞ Norms

The set of all 2D vectors with norm exactly 1 looks completely different
depending on which norm you use. This shape difference is the geometric root of
why L1 vs. L2 regularization behave so differently in practice.

In [ ]:
theta = torch.linspace(0, 2 * torch.pi, 200)

# L2 unit ball: circle, x^2 + y^2 = 1
l2_x, l2_y = torch.cos(theta), torch.sin(theta)

# L1 unit ball: diamond, |x| + |y| = 1  -- parametrize via the same angle, then rescale to norm 1
raw_x, raw_y = torch.cos(theta), torch.sin(theta)
l1_norm = torch.abs(raw_x) + torch.abs(raw_y)
l1_x, l1_y = raw_x / l1_norm, raw_y / l1_norm

# Linf unit ball: square, max(|x|,|y|) = 1
linf_norm = torch.maximum(torch.abs(raw_x), torch.abs(raw_y))
linf_x, linf_y = raw_x / linf_norm, raw_y / linf_norm

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(l2_x, l2_y, label='L2 unit ball (circle)', color='steelblue', lw=2)
ax.plot(l1_x, l1_y, label='L1 unit ball (diamond)', color='darkorange', lw=2)
ax.plot(linf_x, linf_y, label='L∞ unit ball (square)', color='seagreen', lw=2)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_aspect('equal'); ax.legend()
ax.set_title('Unit balls: every point on each curve has norm = 1\n(just measured differently)')
plt.show()

# Spot-check: pick one point from each curve and confirm its own norm is really 1
print("L2 point norm:", torch.norm(torch.tensor([l2_x[10], l2_y[10]]), p=2).item())
print("L1 point norm:", torch.norm(torch.tensor([l1_x[10], l1_y[10]]), p=1).item())
print("L∞ point norm:", torch.norm(torch.tensor([linf_x[10], linf_y[10]]), p=float('inf')).item())

### Exercise 1

1. Create two "embedding-like" vectors of dimension 5 (random values are fine).
2. Compute their cosine similarity.
3. Scale one of the vectors by 10x (multiply every element by 10) — does the cosine
   similarity change? Why or why not, given the definition above?
4. Compute the outer product of the two *original* (unscaled) vectors and confirm
   its shape is `(5, 5)`.

In [ ]:
# Your turn
torch.manual_seed(1)
v1 = torch.randn(5)
v2 = torch.randn(5)

# TODO: compute cosine similarity between v1 and v2
# TODO: scale v1 by 10x and recompute cosine similarity -- compare
# TODO: compute the outer product of v1 and v2, print its shape


---
## Module 2 — Matrices: Multiplication, Transpose, Trace, Determinant, Orthogonality

### Math Basics

**Matrix multiplication as a linear transformation — not just an algorithm.**
This is the single most important reframe in this whole tutorial: a matrix $A$
is completely described by *where it sends the standard basis vectors*. Each
**column** of $A$ is the image of one basis vector. Multiplying $A\mathbf{x}$
means: take $\mathbf{x}$'s coordinates and combine the columns of $A$ in those
proportions. $AB$ **composes** two transformations — apply $B$ first, then $A$:
$(AB)\mathbf{x} = A(B\mathbf{x})$.

**Matrix multiplication vs. Hadamard (element-wise) product.** Matmul *composes
transformations*. The Hadamard product $A \odot B$ has **no transformation
interpretation at all** — it independently rescales each entry, which is why it
shows up as a "gate" or "mask" operation (attention masks, LSTM gates) rather
than a geometric one.

**Trace.** $\text{tr}(A) = \sum_i A_{ii}$, with the cyclic property
$\text{tr}(ABC) = \text{tr}(CAB) = \text{tr}(BCA)$. Trace is invariant under
change of basis — $\text{tr}(P^{-1}AP) = \text{tr}(A)$, a direct consequence of
the cyclic property — meaning trace is a property of the *transformation itself*,
not of the coordinate system used to describe it. It also equals the sum of the
eigenvalues (Module 5).

**Frobenius norm** as $\|A\|_F = \sqrt{\text{tr}(A^\top A)}$ — the L2 norm of the
matrix's entries, all flattened into one long vector.

**Determinant — the geometric meaning is the whole point, not a footnote.**

$$
\det(A) = \text{signed scaling factor of volume under a 3D transformation}
$$

Apply a 3D matrix $A$ to the unit cube; the volume of the resulting
parallelepiped is exactly $|\det(A)|$.

- **Sign matters**: positive $\det(A)$ preserves orientation; negative $\det(A)$
  means the transformation includes a **reflection** (flips orientation, like a
  mirror)
- $\det(A) = 0$ means the transformation **collapses space into a lower
  dimension** — e.g. squashing 3D space onto a plane or line — which is
  exactly why the matrix can't be inverted: an entire dimension of information
  has been destroyed and can't be recovered

**Matrix inverse** exists precisely when $\det(A) \neq 0$ — i.e., when no
dimension gets collapsed, so the transformation can be "undone." Composition
makes this precise: $A^{-1}A = I$ and $AA^{-1} = I$, so applying $A$ and then
$A^{-1}$ returns every point to where it started. For composed transforms, the
inverse runs the steps backward:

$$
(AB)^{-1} = B^{-1}A^{-1}
$$

because $AB\mathbf{x}$ means apply $B$ first, then $A$.

**Transpose.** $A^\top$ swaps rows and columns. For a general matrix this is not
the same as an inverse, but for a rotation/reflection matrix it is: the columns
are orthonormal basis directions, so transposing turns those basis directions
back into coordinate measurements.

**Orthogonal matrices.** $Q^\top Q = I$, equivalently $Q^\top = Q^{-1}$.
Geometrically, orthogonal matrices are **pure rotations and/or reflections — no
stretching at all**. This follows directly from $Q^\top Q = I$: for any vectors
$\mathbf{x}, \mathbf{y}$,

$$
(Q\mathbf{x})\cdot(Q\mathbf{y}) = \mathbf{x}^\top Q^\top Q \mathbf{y} = \mathbf{x}^\top\mathbf{y} = \mathbf{x}\cdot\mathbf{y}
$$

so *every* length and angle is exactly preserved. This is a direct preview of
Module 5: applying an orthogonal matrix to the unit sphere gives back a sphere
(just rotated/reflected); applying a *general* matrix gives an ellipsoid — and the amount of
stretch along each axis of that ellipsoid is exactly what SVD's singular values
measure.

### PyTorch Implementation

In [ ]:
A = torch.tensor([[1.0, 2.0, 0.0],
                  [0.0, 1.0, 3.0],
                  [2.0, 0.0, 1.0]])
B = torch.tensor([[2.0, 0.0, 1.0],
                  [1.0, 3.0, 0.0],
                  [0.0, 2.0, 4.0]])

print("A @ B (matrix mult):\n", A @ B)
print("A * B (Hadamard / element-wise):\n", A * B)
print("A transposed:\n", A.T)

# Trace and Frobenius norm
tr_A = torch.trace(A)
fro_A = torch.norm(A, 'fro')
print("\ntrace(A):", tr_A.item())
print("Frobenius norm of A:", fro_A.item())
print("sqrt(trace(A.T @ A)):", torch.sqrt(torch.trace(A.T @ A)).item())
print("-> these two should match!")

# Determinant and inverse
print("\ndet(A):", torch.linalg.det(A).item())
print("A inverse:\n", torch.linalg.inv(A))

# A singular 3D matrix (the third row is all zeros -> det = 0)
S = torch.tensor([[1.0, 2.0, 0.0],
                  [0.0, 1.0, 3.0],
                  [0.0, 0.0, 0.0]])
print("\ndet(S) for a singular matrix:", torch.linalg.det(S).item())

### Geometric Showcase — A 3D Matrix as a Transformation of Shapes

Instead of thinking of matrix multiplication as "an algorithm that produces
numbers," watch what a handful of common 3x3 matrices actually *do* to a cube.
Each column of the matrix becomes visible as where a basis vector lands.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def unit_cube():
    return torch.tensor([
        [0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0], [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 1.0], [0.0, 1.0, 1.0],
    ])

cube_edges = [(0, 1), (1, 2), (2, 3), (3, 0),
              (4, 5), (5, 6), (6, 7), (7, 4),
              (0, 4), (1, 5), (2, 6), (3, 7)]

def draw_wire_cube(ax, points, color, label=None, linestyle='-', linewidth=2):
    for edge_i, (i, j) in enumerate(cube_edges):
        p, q = points[i], points[j]
        ax.plot([p[0], q[0]], [p[1], q[1]], [p[2], q[2]],
                color=color, linestyle=linestyle, linewidth=linewidth,
                label=label if edge_i == 0 else None)

def plot_transform_3d(ax, M, title):
    cube = unit_cube()
    transformed = cube @ M.T  # row-vector convention: (Mx)^T = x^T M^T
    draw_wire_cube(ax, cube, 'lightgray', label='original', linestyle='--', linewidth=1.5)
    draw_wire_cube(ax, transformed, 'steelblue', label='transformed', linewidth=2.2)

    colors = ['darkorange', 'seagreen', 'crimson']
    labels = [r'$A e_1$', r'$A e_2$', r'$A e_3$']
    for j, (color, label) in enumerate(zip(colors, labels)):
        col = M[:, j]
        ax.quiver(0, 0, 0, col[0], col[1], col[2], color=color,
                  linewidth=2.5, arrow_length_ratio=0.12, label=label)

    all_points = torch.cat([cube, transformed], dim=0)
    mins = all_points.min(dim=0).values - 0.4
    maxs = all_points.max(dim=0).values + 0.4
    ax.set_xlim(mins[0], maxs[0]); ax.set_ylim(mins[1], maxs[1]); ax.set_zlim(mins[2], maxs[2])
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=22, azim=-45)

scale_mat = torch.diag(torch.tensor([1.8, 0.7, 1.3]))
shear_mat = torch.tensor([[1.0, 0.7, 0.0],
                          [0.0, 1.0, 0.4],
                          [0.0, 0.0, 1.0]])
angle = torch.tensor(0.6)
rotate_mat = torch.tensor([[torch.cos(angle).item(), -torch.sin(angle).item(), 0.0],
                           [torch.sin(angle).item(),  torch.cos(angle).item(), 0.0],
                           [0.0,                      0.0,                     1.0]])
reflect_mat = torch.diag(torch.tensor([1.0, 1.0, -1.0]))

fig = plt.figure(figsize=(18, 5.5))
for idx, (name, M) in enumerate([('Scale', scale_mat), ('Shear', shear_mat),
                                ('Rotate around z', rotate_mat), ('Reflect z', reflect_mat)], start=1):
    ax = fig.add_subplot(1, 4, idx, projection='3d')
    plot_transform_3d(ax, M, f"{name}\ndet={torch.linalg.det(M).item():.2f}")
    if idx == 1:
        ax.legend(loc='upper left', fontsize=8)

plt.suptitle('Columns of a 3x3 matrix show where the three 3D basis vectors land')
plt.tight_layout()
plt.show()

print('Rotation preserves volume and shape. Reflection preserves volume but flips orientation.')
print('Scale changes volume; shear keeps volume here because det(shear_mat) = 1.')

### Geometric Showcase — Determinant as Signed Volume

Let's directly verify that $|\det(A)|$ equals the volume of the parallelepiped
formed by transforming the unit cube, and confirm the sign flips under a
reflection. We'll also apply $A^{-1}$ to recover the original cube whenever the
matrix is invertible, then compose two transforms and undo the composition in
reverse order.

In [ ]:
def parallelepiped_volume(M):
    # The transformed unit cube is spanned by the columns of M: A e1, A e2, A e3.
    c1, c2, c3 = M[:, 0], M[:, 1], M[:, 2]
    signed_volume = torch.dot(c1, torch.cross(c2, c3, dim=0))
    return signed_volume, torch.abs(signed_volume)

cube = unit_cube()
for name, M in [("scale", scale_mat), ("shear", shear_mat),
                 ("rotate", rotate_mat), ("reflect", reflect_mat)]:
    signed_volume, volume = parallelepiped_volume(M)
    det = torch.linalg.det(M)
    M_inv = torch.linalg.inv(M)
    transformed = cube @ M.T
    recovered = transformed @ M_inv.T
    print(f"{name:8s} | volume: {volume.item():.4f} | |det(A)|: {torch.abs(det).item():.4f} "
          f"| det(A): {det.item():+.4f} | recovered by A^-1: {torch.allclose(recovered, cube, atol=1e-5)}")

print("\nVolume always matches |det(A)|. A negative determinant means orientation flipped.")
print("For every invertible example above, A^-1 maps the transformed cube back to the original cube.")

# Composition: C = shear @ rotate means rotate first, then shear (column-vector convention).
C = shear_mat @ rotate_mat
C_inv = torch.linalg.inv(C)
rotate_inv = torch.linalg.inv(rotate_mat)
shear_inv = torch.linalg.inv(shear_mat)

cube_rotated = cube @ rotate_mat.T
cube_rotated_then_sheared = cube_rotated @ shear_mat.T
cube_composed = cube @ C.T
cube_recovered = cube_composed @ C_inv.T

print("\ncomposition check:")
print("C = shear @ rotate applies rotate first, then shear:",
      torch.allclose(cube_rotated_then_sheared, cube_composed, atol=1e-5))
print("C_inv @ C is identity:\n", C_inv @ C)
print("inverse of shear @ rotate equals rotate^-1 @ shear^-1:",
      torch.allclose(C_inv, rotate_inv @ shear_inv, atol=1e-5))
print("composed inverse recovers the cube:", torch.allclose(cube_recovered, cube, atol=1e-5))

fig = plt.figure(figsize=(6.5, 6))
ax = fig.add_subplot(1, 1, 1, projection='3d')
draw_wire_cube(ax, cube, 'lightgray', label='original', linestyle='--', linewidth=1.5)
draw_wire_cube(ax, cube_composed, 'darkorange', label='C @ cube', linewidth=2.2)
draw_wire_cube(ax, cube_recovered, 'seagreen', label='C^-1 @ C @ cube', linewidth=1.8)
all_points = torch.cat([cube, cube_composed, cube_recovered], dim=0)
mins = all_points.min(dim=0).values - 0.4
maxs = all_points.max(dim=0).values + 0.4
ax.set_xlim(mins[0], maxs[0]); ax.set_ylim(mins[1], maxs[1]); ax.set_zlim(mins[2], maxs[2])
ax.set_box_aspect((1, 1, 1))
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Composed transform and inverse recovery')
ax.legend(loc='upper left', fontsize=8)
ax.view_init(elev=22, azim=-45)
plt.tight_layout()
plt.show()

### Geometric Showcase — Orthogonal Matrices Preserve the Unit Sphere

An orthogonal matrix should map the unit sphere to... the unit sphere (just
possibly rotated/reflected), since it can't stretch anything. A *general* matrix,
by contrast, turns the sphere into an ellipsoid. Applying the inverse of that
general matrix maps the ellipsoid back to the sphere. This is the direct setup
for SVD in Module 5, where the amount of that stretching becomes the singular
values.

In [ ]:
# Orthogonal matrix example: a 3D rotation around the z-axis is always orthogonal
theta = torch.tensor(0.7)
Q = torch.tensor([
    [torch.cos(theta).item(), -torch.sin(theta).item(), 0.0],
    [torch.sin(theta).item(),  torch.cos(theta).item(), 0.0],
    [0.0,                      0.0,                     1.0],
])

print("Q:\n", Q)
print("Q.T @ Q (should be ~identity):\n", Q.T @ Q)
print("Q inverse:\n", torch.linalg.inv(Q))
print("Q transpose (should match inverse above):\n", Q.T)
print("Q.T equals Q inverse:", torch.allclose(Q.T, torch.linalg.inv(Q), atol=1e-5))

# Use transpose as the inverse: rotate a vector, then un-rotate it with Q.T.
v = torch.tensor([1.0, 0.2, 0.5])
v_rotated = Q @ v
v_recovered_by_transpose = Q.T @ v_rotated

print("\noriginal v:", v)
print("Q @ v:", v_rotated)
print("Q.T @ (Q @ v):", v_recovered_by_transpose)
print("transpose undoes this rotation:", torch.allclose(v_recovered_by_transpose, v, atol=1e-5))

In [ ]:
phi = torch.linspace(0, torch.pi, 40)
theta_grid = torch.linspace(0, 2 * torch.pi, 80)
Phi, Theta = torch.meshgrid(phi, theta_grid, indexing='ij')
sphere_x = torch.sin(Phi) * torch.cos(Theta)
sphere_y = torch.sin(Phi) * torch.sin(Theta)
sphere_z = torch.cos(Phi)
sphere_pts = torch.stack([sphere_x.reshape(-1), sphere_y.reshape(-1), sphere_z.reshape(-1)], dim=1)

Q_transformed = sphere_pts @ Q.T
general_mat = torch.tensor([[1.8, 0.5, 0.0],
                            [0.2, 0.9, 0.4],
                            [0.0, 0.3, 1.2]])  # NOT orthogonal
general_transformed = sphere_pts @ general_mat.T
recovered_sphere = general_transformed @ torch.linalg.inv(general_mat).T

def as_surface(points):
    return [points[:, i].reshape(Phi.shape).numpy() for i in range(3)]

fig = plt.figure(figsize=(16, 5.5))
axes = [fig.add_subplot(1, 3, i, projection='3d') for i in range(1, 4)]
for ax, pts, title, color in [
    (axes[0], sphere_pts, 'Unit sphere', 'lightgray'),
    (axes[1], Q_transformed, 'Orthogonal Q: sphere stays a sphere', 'steelblue'),
    (axes[2], general_transformed, 'General A: sphere becomes an ellipsoid', 'darkorange'),
]:
    Xs, Ys, Zs = as_surface(pts)
    ax.plot_surface(Xs, Ys, Zs, color=color, alpha=0.55, linewidth=0, shade=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_zlim(-2.2, 2.2)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.view_init(elev=22, azim=-45)

plt.tight_layout()
plt.show()

print('Q preserves every radius:', torch.allclose(torch.linalg.norm(Q_transformed, dim=1),
                                                  torch.ones(len(Q_transformed)), atol=1e-5))
print('A is invertible, det(A) =', torch.linalg.det(general_mat).item())
print('A^-1 recovers the original sphere:', torch.allclose(recovered_sphere, sphere_pts, atol=1e-5))

### Exercise 2

1. Verify the Frobenius-norm-via-trace identity `||A||_F == sqrt(trace(A.T @ A))`
   on a **random** 3x3 matrix (not the example above).
2. Confirm the cyclic trace property `trace(ABC) == trace(CAB) == trace(BCA)` on
   three random 3x3 matrices.
3. Build any 3D rotation matrix of your choice and confirm it's orthogonal by
   checking `Q.T @ Q` is (approximately) the identity matrix.

In [ ]:
# Your turn
torch.manual_seed(2)
# TODO: create a random 3x3 matrix and verify the Frobenius norm / trace identity

# TODO: create three random 3x3 matrices A, B, C and verify the cyclic trace property

# TODO: build a 3D rotation matrix with an angle of your choice and confirm orthogonality


---
## Module 3 — Vectorization, Broadcasting, Reshaping, and Batched Operations

### Math Basics

**Vectorization** means replacing an explicit element-by-element (or row-by-row)
Python `for` loop with a single tensor-level operation that applies across an
entire axis at once. This is the single most important practical habit for
writing fast PyTorch code.

The key conceptual point: vectorization is **not a different computation** — a
loop and its vectorized equivalent compute exactly the same numbers. What changes
is *how* the computation is carried out, not *what* it computes.

**Why vectorized code is faster:** a Python loop pays interpreter overhead on
every single iteration; a vectorized op dispatches the *entire* computation in
one call to optimized, contiguous, parallelized C/CUDA code underneath PyTorch.
Same math, radically different execution cost.

**Broadcasting** is the mechanism that makes a lot of vectorization possible: it
lets you operate on tensors of different shapes without manually copying data —
smaller tensors are conceptually "stretched" to match larger ones, following
alignment rules from the *trailing* dimension inward.

**Reshaping** rearranges elements into a new shape *without changing the total
element count* — that invariant is the first thing to check when debugging a
reshape error.

**Batched operations** extend the same idea to matrix multiplication: `torch.bmm`
performs the same matrix multiply across a batch dimension *in one call*, rather
than looping over each matrix pair. `torch.einsum` generalizes this (and nearly
every other tensor contraction) using Einstein summation notation.

### Worked Example — For-Loop vs. Vectorized: Row-Wise L2 Normalization

Given `X` of shape `(N, C)` — `N` samples, each with `C` features — normalize
every **row** to unit length:

$$
\mathbf{y}_i = \frac{\mathbf{x}_i}{\|\mathbf{x}_i\|_2} \quad \text{for each row } i = 1, \dots, N
$$

This is a direct callback to Module 1's L2 norm, and it's a genuinely common
operation (e.g. normalizing embeddings before computing cosine similarity — once
vectors are unit-length, their dot product *is* the cosine similarity).

We'll implement this two ways and confirm they give identical results — then time
both to see why vectorized code is the default in deep learning.

In [ ]:
import time

torch.manual_seed(9)
N, C = 10_000, 512
X = torch.randn(N, C)

# ---- Version 1: explicit for-loop, one row at a time ----
start = time.time()
Y_loop = torch.zeros_like(X)
for i in range(N):
    row = X[i]
    row_norm = torch.norm(row)
    Y_loop[i] = row / row_norm
loop_time = time.time() - start

# ---- Version 2: vectorized, all rows at once ----
start = time.time()
row_norms = torch.norm(X, dim=1, keepdim=True)  # shape (N, 1)
Y_vec = X / row_norms                             # broadcasting: (N, C) / (N, 1)
vec_time = time.time() - start

print(f"Loop version:       {loop_time:.4f} sec")
print(f"Vectorized version: {vec_time:.4f} sec")
print(f"Speedup: {loop_time / vec_time:.1f}x")
print("\nResults match:", torch.allclose(Y_loop, Y_vec, atol=1e-5))

# Sanity check: every row of Y_vec should now have norm ~1
print("norm of first 3 normalized rows:", torch.norm(Y_vec[:3], dim=1))

In [ ]:
# Broadcasting example
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)
v = torch.tensor([10.0, 20.0, 30.0])                    # shape (3,)
print("x + v (broadcast over rows):\n", x + v)

col = torch.tensor([[100.0], [200.0]])  # shape (2, 1)
print("\nx + col (broadcast over columns):\n", x + col)

In [ ]:
# Reshaping -- total elements must stay the same
t = torch.arange(24)  # 24 elements
print("original shape:", t.shape)

t_reshaped = t.reshape(2, 3, 4)   # 2*3*4 = 24 -- valid
print("reshaped to (2,3,4):", t_reshaped.shape)

# permute reorders axes (not the same as reshape!)
t_permuted = t_reshaped.permute(2, 0, 1)
print("permuted to (4,2,3):", t_permuted.shape)

# A very common real-world reshape: flattening a batch of images for a linear layer
images = torch.randn(16, 3, 28, 28)  # (batch, channels, height, width)
flattened = images.view(16, -1)       # (batch, channels*height*width)
print("\nimages:", images.shape, "-> flattened:", flattened.shape)

In [ ]:
# Batched matrix multiplication
batch_size = 8
A_batch = torch.randn(batch_size, 3, 4)  # 8 matrices, each 3x4
B_batch = torch.randn(batch_size, 4, 5)  # 8 matrices, each 4x5

# Using bmm
result_bmm = torch.bmm(A_batch, B_batch)
print("bmm result shape:", result_bmm.shape)  # (8, 3, 5)

# The same thing with einsum: 'bij,bjk->bik' means:
#   for each batch b, contract the shared 'j' dimension
result_einsum = torch.einsum('bij,bjk->bik', A_batch, B_batch)
print("einsum result shape:", result_einsum.shape)

print("Results match:", torch.allclose(result_bmm, result_einsum, atol=1e-5))

### Exercise 3

1. Implement a **for-loop** version and a **vectorized** version of row-wise
   **mean** (instead of L2 norm) on a `(N, C)` tensor with `N=5000, C=256`.
   Time both, confirm they match, and note the speedup.
2. You have a batch of 32 grayscale images shaped `(32, 1, 64, 64)`. Reshape them
   to `(32, 4096)` (flatten each image into a single vector) using `.view` or
   `.reshape`.
3. Implement a **batched dot product**: given `a` and `b` both of shape
   `(batch, dim)`, compute the dot product *per batch element* (result shape
   `(batch,)`) using `torch.einsum`. Verify against a manual loop for a small batch.
4. Deliberately write a reshape that is *invalid* (wrong total element count),
   run it, and read the error message PyTorch gives you.

In [ ]:
# Your turn
torch.manual_seed(3)

# TODO 1: for-loop vs. vectorized row-wise mean on a (5000, 256) tensor, time both

# TODO 2: reshape a (32, 1, 64, 64) tensor to (32, 4096)

# TODO 3: batched dot product via einsum, verify against a manual loop

# TODO 4: trigger and observe an invalid reshape error


---
## Module 4 — Linear Transformations & `nn.Linear` Demystified

### Math Basics

A **linear transformation** maps an input vector to an output vector via matrix
multiplication, written the way you would on paper — with the vector as a
**column** on the right:

$$\mathbf{y} = W\mathbf{x}$$

Neural network layers use an **affine transformation** (linear + a shift):

$$\mathbf{y} = W\mathbf{x} + \mathbf{b}$$

Read the shapes off the equation: $W$ is $(\text{out} \times \text{in})$,
$\mathbf{x}$ is $(\text{in},)$, and $\mathbf{y}$ is $(\text{out},)$ — the matrix
"eats" the input dimension and leaves the output dimension behind. This is all a
"dense" or "fully-connected" layer is: a matrix multiply plus a bias vector,
nothing more.

> **Note — why PyTorch writes it as $XW^\top + \mathbf{b}$.** Transpose both
> sides of $\mathbf{y} = W\mathbf{x}$ and you get
> $\mathbf{y}^\top = \mathbf{x}^\top W^\top$. PyTorch stores a batch with one
> sample per **row** (the batch dimension always comes first), so it uses that
> transposed form: `nn.Linear` computes $Y = XW^\top + \mathbf{b}$ for
> $X$ of shape $(\text{batch}, \text{in})$. **It is the same equation** — only
> the memory layout differs, row vectors instead of column vectors. We'll write
> $W\mathbf{x} + \mathbf{b}$ below and then show both forms landing on identical
> numbers, so the transposes never become mysterious.

**Geometric picture.** A pure linear map $\mathbf{y} = W\mathbf{x}$ always
keeps the origin fixed — $W\mathbf{0} = \mathbf{0}$, no matter what $W$ is. Adding
the bias $\mathbf{b}$ turns it into an **affine** map: first apply the linear
transformation (rotate/scale/shear the space, as in Module 2), *then* slide the
entire result by $\mathbf{b}$. This is why a network with only linear layers and
no bias can never shift a decision boundary away from the origin — the bias
vector is what buys that freedom.

### PyTorch Implementation

In [ ]:
import torch.nn as nn

torch.manual_seed(4)

in_features, out_features = 5, 3
batch_size = 2

x = torch.randn(batch_size, in_features)

# The "official" way
linear_layer = nn.Linear(in_features, out_features)
y_official = linear_layer(x)
print("nn.Linear output:\n", y_official)
print("weight shape:", linear_layer.weight.shape)  # (out_features, in_features)
print("bias shape:", linear_layer.bias.shape)        # (out_features,)

# A single sample (no batch dimension) works too -- this is the plain y = Wx + b case
y_single = linear_layer(x[0])
print("\nsingle-sample output:", y_single.detach(), "| shape:", y_single.shape)

In [ ]:
# Reimplementing the layer by hand, in the textbook form y = Wx + b
W = linear_layer.weight   # (out_features, in_features)
b = linear_layer.bias     # (out_features,)

# --- One sample: a column vector in, a column vector out ---
x0 = x[0]                 # (in_features,) -- treated as a column vector
y0 = W @ x0 + b           # (out_features,)
print("y = Wx + b   ->", y0.detach())
print("nn.Linear    ->", linear_layer(x0).detach())
print("match:", torch.allclose(linear_layer(x0), y0, atol=1e-6))

# --- A whole batch, still y = Wx + b: stack the samples as COLUMNS ---
X_cols = x.T                          # (in_features, batch) -- one sample per column
Y_cols = W @ X_cols + b.unsqueeze(1)  # (out_features, batch); bias broadcasts across columns
print("\ncolumn-major batch shapes:", X_cols.shape, "->", Y_cols.shape)
print("match:", torch.allclose(y_official, Y_cols.T, atol=1e-6))  # .T back to (batch, out)

# --- The same thing in the row-major form PyTorch actually uses ---
# Transpose both sides of y = Wx:  (Wx)^T = x^T W^T.  So if your samples are ROWS
# (as they are everywhere in PyTorch, because a batch dimension comes first), the
# identical computation is written X W^T + b -- no math has changed, only the layout.
y_rows = x @ W.T + b
print("\nrow-major form X W^T + b matches:", torch.allclose(y_official, y_rows, atol=1e-6))

### Exercise 4

1. Create your own `nn.Linear(4, 2)` layer.
2. Manually implement the same computation with raw tensor ops and the layer's
   own `.weight` / `.bias`, in the form $\mathbf{y} = W\mathbf{x} + \mathbf{b}$
   — first for a single sample, then for a whole batch with the samples as
   columns (remember to transpose the result back to `(batch, out)` before
   comparing).
3. Confirm your manual output matches the layer's output using `torch.allclose`.
4. Stack **two** linear layers manually (with a `torch.relu` in between) to
   reproduce a tiny 2-layer MLP forward pass, and compare against
   `nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,2))`. Keep the samples
   in columns the whole way through, then check that the row-major form
   $XW^\top + \mathbf{b}$ gives the same answer.

In [ ]:
# Your turn
torch.manual_seed(5)

# TODO 1-3: single linear layer, manual reimplementation, allclose check

# TODO 4: two-layer MLP, manual vs nn.Sequential


---
## Module 5 — Rank, Eigenvalues, SVD, PCA, and Low-Rank Approximation

### Math Basics

**Rank.** The number of linearly independent rows/columns of a matrix.
Geometrically, rank is **the dimension of the space the transformation actually
reaches** — a rank-1 matrix (like the outer products from Module 1) squashes
every input onto a single line; a rank-2 matrix onto a plane; and so on. A matrix
is **full rank** if its rank equals `min(rows, cols)`, otherwise it's
**rank-deficient**.

**Eigenvalues/eigenvectors — the geometric picture *is* the definition.** For a
square matrix $A$, an eigenvector $\mathbf{v}$ satisfies

$$
A\mathbf{v} = \lambda\mathbf{v}
$$

meaning: applying $A$ to $\mathbf{v}$ **does not rotate it at all** — it only
stretches or shrinks $\mathbf{v}$ by the factor $\lambda$. Every *other* vector
gets both rotated and scaled by $A$, but eigenvectors only get scaled. Picture a
circle of points transformed into an ellipse by $A$: the eigenvectors are exactly
the directions pointing along that ellipse's axes.

**SVD (Singular Value Decomposition).** Any matrix — square or not, symmetric or
not — factors as

$$
A = U \Sigma V^\top
$$

where $U, V$ are orthogonal and $\Sigma$ is diagonal with non-negative singular
values in decreasing order. Geometrically, this is a **three-step decomposition
of any transformation**:

$$
\text{rotate } (V^\top) \;\rightarrow\; \text{scale along axes } (\Sigma) \;\rightarrow\; \text{rotate } (U)
$$

Concretely: $A$ maps the unit circle to an ellipse. The **singular values**
(entries of $\Sigma$) are the lengths of that ellipse's semi-axes, and the
columns of $U$ give the ellipse's final orientation in space. SVD is the natural
generalization of eigenvectors to matrices where a single "this direction doesn't
rotate" story doesn't apply (non-square or non-symmetric matrices) — instead it
says "these input directions map to these output directions, stretched by these
amounts."

**Variance explained via trace**: top-$k$ components capture
$\text{tr}(\Sigma_k) / \text{tr}(\Sigma)$ of total variance (trace callback #2).

**Low-rank approximation.** Keeping only the top-$k$ singular values/vectors
gives the *best possible* rank-$k$ approximation of a matrix — using far fewer
numbers to represent nearly the same information. This is the core idea behind
**LoRA (Low-Rank Adaptation)**, a popular technique for efficiently fine-tuning
large models by learning small low-rank updates instead of full weight matrices.

### PyTorch Implementation

In [ ]:
# Rank
full_rank_mat = torch.randn(4, 4)
rank_deficient_mat = torch.tensor([[1.0, 2.0, 3.0],
                                    [2.0, 4.0, 6.0],   # this row is 2x the first
                                    [0.0, 1.0, 1.0]])

print("rank of random 4x4:", torch.linalg.matrix_rank(full_rank_mat).item())
print("rank of rank-deficient 3x3:", torch.linalg.matrix_rank(rank_deficient_mat).item())

In [ ]:
# Eigenvalues / eigenvectors of a symmetric matrix
S = torch.tensor([[4.0, 1.0], [1.0, 3.0]])
eigvals, eigvecs = torch.linalg.eig(S)
print("eigenvalues:", eigvals)
print("eigenvectors:\n", eigvecs)

# SVD
M = torch.randn(5, 3)
U, Sigma, Vt = torch.linalg.svd(M, full_matrices=False)
print("\nU:", U.shape, "| Sigma:", Sigma.shape, "| Vt:", Vt.shape)

# Reconstruct M from its SVD components
M_reconstructed = U @ torch.diag(Sigma) @ Vt
print("reconstruction matches original:", torch.allclose(M, M_reconstructed, atol=1e-5))

### Geometric Showcase — Eigenvectors as the Axes of a Transformed Ellipse

Transform a circle of points by a symmetric matrix $S$ and watch it become an
ellipse. The eigenvectors point exactly along that ellipse's axes, and the
eigenvalues tell you how much each axis got stretched.

In [ ]:
S_demo = torch.tensor([[3.0, 1.0], [1.0, 2.0]])  # symmetric -> real eigenvalues/orthogonal eigenvectors
eigvals_demo, eigvecs_demo = torch.linalg.eigh(S_demo)  # eigh: for symmetric matrices

circle_theta = torch.linspace(0, 2 * torch.pi, 100)
circle_pts = torch.stack([torch.cos(circle_theta), torch.sin(circle_theta)], dim=1)
ellipse_pts = circle_pts @ S_demo.T

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(circle_pts[:, 0], circle_pts[:, 1], '--', color='lightgray', label='unit circle (input)')
ax.plot(ellipse_pts[:, 0], ellipse_pts[:, 1], color='steelblue', lw=2, label='S @ circle (output)')

# Draw eigenvectors, scaled by their eigenvalues -- these should land exactly on the ellipse's axes
for i in range(2):
    v = eigvecs_demo[:, i] * eigvals_demo[i]
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
               color='darkorange' if i == 0 else 'seagreen', width=0.02,
               label=f'eigenvector {i+1} (λ={eigvals_demo[i].item():.2f})')

ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_aspect('equal'); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.legend(loc='upper left', fontsize=9)
ax.set_title('Eigenvectors point exactly along the axes of the transformed ellipse')
plt.show()

print("eigenvalues:", eigvals_demo)
print("Notice both eigenvectors land EXACTLY on the ellipse boundary -- they're the only")
print("directions that got purely scaled (not rotated) by S.")

### Geometric Showcase — SVD as Rotate → Scale → Rotate

Now watch SVD's three-step decomposition happen live: start with the unit
circle, apply $V^\top$ (a rotation — still a circle), apply $\Sigma$ (scales
along the axes — now an axis-aligned ellipse), then apply $U$ (a final rotation
— the ellipse's final orientation). The end result should exactly match applying
$A$ directly.

In [ ]:
A_svd = torch.tensor([[1.5, 0.8], [0.3, 1.2]])  # a general (non-symmetric) matrix
U_svd, S_svd, Vt_svd = torch.linalg.svd(A_svd)

step0 = circle_pts                              # start: unit circle
step1 = circle_pts @ Vt_svd.T                    # apply V^T: rotate
step2 = step1 @ torch.diag(S_svd).T              # apply Sigma: scale along axes
step3 = step2 @ U_svd.T                          # apply U: rotate again
direct = circle_pts @ A_svd.T                    # sanity check: direct A @ circle

fig, axes = plt.subplots(1, 4, figsize=(19, 5))
titles = ["Step 0: unit circle", "Step 1: after $V^T$ (rotate)",
          "Step 2: after $\\Sigma$ (scale)", "Step 3: after $U$ (rotate) = final"]
for ax, pts, title in zip(axes, [step0, step1, step2, step3], titles):
    ax.plot(pts[:, 0], pts[:, 1], color='steelblue', lw=2)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal'); ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()

print("singular values (ellipse semi-axis lengths):", S_svd)
print("Step 3 matches direct A @ circle:", torch.allclose(step3, direct, atol=1e-5))

In [ ]:
# PCA via SVD on real data (sklearn's digits dataset)
from sklearn.datasets import load_digits

digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32)   # shape (n_samples, 64)
y = digits.target

# Mean-center the data (required for PCA)
X_centered = X - X.mean(dim=0, keepdim=True)

U, S, Vt = torch.linalg.svd(X_centered, full_matrices=False)

# Project onto the top 2 principal components for visualization
X_2d = X_centered @ Vt[:2].T   # shape (n_samples, 2)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', s=10)
plt.colorbar(scatter, label='digit label')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('PCA of digits dataset (via SVD) -- top 2 components')
plt.show()

# Variance explained via trace: tr(Sigma_k) / tr(Sigma)
# For SVD of a centered data matrix, singular values relate to the covariance eigenvalues
total_variance = torch.sum(S ** 2)
var_explained_2 = torch.sum(S[:2] ** 2) / total_variance
print(f"Variance explained by top 2 components: {var_explained_2.item():.2%}")

In [ ]:
# Low-rank approximation demo: compress a random "image" matrix
torch.manual_seed(6)
img_matrix = torch.randn(50, 50)
# Make it have real structure (low effective rank) by constructing it as a product
true_rank = 5
img_matrix = torch.randn(50, true_rank) @ torch.randn(true_rank, 50)

U_img, S_img, Vt_img = torch.linalg.svd(img_matrix, full_matrices=False)

errors = []
ks = list(range(1, 21))
for k in ks:
    approx = U_img[:, :k] @ torch.diag(S_img[:k]) @ Vt_img[:k, :]
    error = torch.norm(img_matrix - approx, 'fro').item()
    errors.append(error)

plt.figure(figsize=(6, 4))
plt.plot(ks, errors, marker='o')
plt.axvline(true_rank, color='red', linestyle='--', label=f'true rank = {true_rank}')
plt.xlabel('k (number of singular values kept)')
plt.ylabel('Frobenius norm reconstruction error')
plt.title('Low-rank approximation error vs. k')
plt.legend()
plt.show()
print("Notice how error collapses to ~0 right at the true rank -- this is the core"
      " idea behind LoRA-style low-rank weight updates.")

### Exercise 5

1. Compute the rank of a matrix built as `torch.randn(6, 2) @ torch.randn(2, 6)`
   (a 6x6 matrix). What do you expect the rank to be, and why?
2. Using the digits PCA setup above, compute variance explained by the top **5**
   components (instead of 2) using the trace-based formula.
3. Reconstruct the digits data matrix `X_centered` using only the **top 10**
   singular values/vectors, and compute the Frobenius norm of the reconstruction
   error.

In [ ]:
# Your turn
torch.manual_seed(7)

# TODO 1: rank of a deliberately low-rank 6x6 matrix

# TODO 2: variance explained by top 5 PCA components on the digits data

# TODO 3: reconstruct X_centered using top-10 SVD components, measure error


---
## Module 6 — Gradients, Jacobians, Autograd, and Positive Definiteness

### Math Basics

**Gradient as a vector — the geometric meaning.** For a scalar-valued function
$f(\mathbf{x})$, the gradient $\nabla f$ points in the direction of **steepest
ascent**, and its magnitude is the rate of increase in that direction. A
crucial, often-missed fact: the gradient is always **perpendicular (orthogonal)
to the level curves/contours** of $f$. If you stand on a contour line of
constant height and ask "which way increases height fastest," the answer is
always straight across the contour — never along it.

**Jacobian as a matrix.** For a vector-valued function $\mathbf{f}(\mathbf{x})$,
the Jacobian is the natural generalization of the gradient: row $i$ of the
Jacobian is the gradient of output $i$ with respect to all inputs.

**Chain rule in matrix form** is what makes backpropagation work: gradients
through a composition of layers are computed as products of Jacobians.

### PyTorch Implementation

In [ ]:
# Basic autograd: gradient of a scalar function
x = torch.tensor([2.0, 3.0], requires_grad=True)
f = (x ** 2).sum()  # f(x) = x1^2 + x2^2
f.backward()
print("x:", x.detach())
print("gradient of f w.r.t. x:", x.grad)          # expected: [2*x1, 2*x2] = [4, 6]

### Geometric Showcase — Gradient as Steepest Ascent, Perpendicular to Contours

Take $f(x, y) = x^2 + 2y^2$ (an elliptical "bowl"). At several sample points,
plot the gradient as an arrow and overlay it on the function's contour lines.
The arrows should always point directly away from the center (steepest ascent)
and always cross the contour lines at a right angle.

In [ ]:
def f_bowl(x, y):
    return x**2 + 2 * y**2

def grad_f_bowl(x, y):
    return torch.tensor([2 * x, 4 * y])  # analytical gradient of x^2 + 2y^2

xs = torch.linspace(-3, 3, 200)
ys = torch.linspace(-3, 3, 200)
X_grid, Y_grid = torch.meshgrid(xs, ys, indexing='xy')
Z_grid = f_bowl(X_grid, Y_grid)

fig, ax = plt.subplots(figsize=(7, 7))
contours = ax.contour(X_grid, Y_grid, Z_grid, levels=12, cmap='Blues')
ax.clabel(contours, inline=True, fontsize=7)

sample_points = [(-2.0, -1.5), (2.0, 1.0), (-1.0, 2.0), (1.5, -2.0), (0.5, 0.5)]
for (px, py) in sample_points:
    g = grad_f_bowl(torch.tensor(px), torch.tensor(py))
    g_normalized = g / torch.norm(g) * 0.6  # normalize arrow length for clean plotting
    ax.quiver(px, py, g_normalized[0], g_normalized[1], angles='xy', scale_units='xy',
               scale=1, color='darkorange', width=0.012)
    ax.plot(px, py, 'o', color='darkorange', markersize=5)

ax.set_aspect('equal')
ax.set_title(r'$f(x,y)=x^2+2y^2$: gradient arrows point uphill,'
             '\nalways perpendicular to the contour lines')
plt.show()

### Worked Example — Gradient of a Linear (MLP-like) Layer $y = Ax$

This is the simplest possible version of what backprop computes at *every* layer
of a neural network — including every layer in the capstone MLP below. Working
through it by hand makes autograd feel far less like a black box.

**Setup** (same MSE pattern used throughout this tutorial, including the capstone):

$$
\mathbf{x} \in \mathbb{R}^n, \quad A \in \mathbb{R}^{m \times n}, \quad
\mathbf{y} = A\mathbf{x} \in \mathbb{R}^m, \quad \mathbf{t} \in \mathbb{R}^m
$$

$$
L = \tfrac{1}{2}\|\mathbf{y} - \mathbf{t}\|^2 = \tfrac{1}{2}\|A\mathbf{x} - \mathbf{t}\|^2
$$

Define the **error vector** $\mathbf{e} = A\mathbf{x} - \mathbf{t} = \mathbf{y} - \mathbf{t}$.

**Gradient with respect to the input** $\mathbf{x}$:

$$
\frac{\partial L}{\partial \mathbf{x}} = A^\top \mathbf{e} = A^\top(A\mathbf{x} - \mathbf{t})
$$

This is a matrix-vector product using $A^\top$ — literally what "backprop through
a linear layer" means: take the incoming gradient and multiply by the transpose
of the weight matrix.

**Gradient with respect to the weights** $A$:

$$
\frac{\partial L}{\partial A} = \mathbf{e}\,\mathbf{x}^\top = (A\mathbf{x} - \mathbf{t})\,\mathbf{x}^\top
$$

This is an **outer product** of the error vector and the input vector — a direct
callback to Module 1. "Outer product of error and input" is *the* formula behind
every weight update in gradient descent for a linear layer, and it recurs
throughout deep learning (it's exactly what happens inside `loss.backward()` for
every `nn.Linear` in a real network).

Let's verify both closed-form results numerically against `torch.autograd`.

In [ ]:
torch.manual_seed(11)

n, m = 4, 3  # input dim, output dim
A_ex = torch.randn(m, n, requires_grad=True)
x_ex = torch.randn(n, requires_grad=True)
t_ex = torch.randn(m)

y_ex = A_ex @ x_ex
e_ex = y_ex - t_ex
L_ex = 0.5 * torch.sum(e_ex ** 2)

L_ex.backward()

# Closed-form gradients derived above
grad_A_closed_form = torch.outer(e_ex.detach(), x_ex.detach())  # e . x^T
grad_x_closed_form = A_ex.detach().T @ e_ex.detach()             # A^T . e

print("dL/dA (autograd):\n", A_ex.grad)
print("dL/dA (closed form, outer(e, x)):\n", grad_A_closed_form)
print("match:", torch.allclose(A_ex.grad, grad_A_closed_form, atol=1e-5))

print("\ndL/dx (autograd):", x_ex.grad)
print("dL/dx (closed form, A.T @ e):", grad_x_closed_form)
print("match:", torch.allclose(x_ex.grad, grad_x_closed_form, atol=1e-5))

In [ ]:
# Jacobian of a vector-valued function
def vector_fn(x):
    return torch.stack([x[0]**2 + x[1], x[0] * x[1], torch.sin(x[0])])

x2 = torch.tensor([1.0, 2.0])
J = torch.autograd.functional.jacobian(vector_fn, x2)
print("Jacobian:\n", J)  # shape (3, 2) -- 3 outputs, 2 inputs

In [ ]:
# Trace trick example: gradient of f(W) = trace(W.T @ W) w.r.t. W
# By the trace trick, d/dW trace(W^T W) = 2W -- let's verify with autograd
W = torch.randn(3, 3, requires_grad=True)
f_trace = torch.trace(W.T @ W)
f_trace.backward()

print("autograd gradient:\n", W.grad)
print("\nanalytical (2W):\n", 2 * W.detach())
print("\nmatch:", torch.allclose(W.grad, 2 * W.detach(), atol=1e-5))

### Geometric Showcase — Positive Definite vs. Indefinite: Bowl vs. Saddle

A symmetric matrix $A$ is **positive definite (PD)** if $\mathbf{x}^\top A
\mathbf{x} > 0$ for every nonzero $\mathbf{x}$ (PSD allows $\geq 0$), which is
equivalent to *all its eigenvalues being positive* (non-negative for PSD).

**Geometric meaning:** plot the quadratic form $z = \mathbf{x}^\top A \mathbf{x}$
as a surface.
- If $A$ is **PD**, every contour is an ellipse and the surface is a **bowl** —
  one clear global minimum at the origin.
- If $A$ has **mixed-sign eigenvalues** (indefinite), the surface is a
  **saddle** — it goes up in some directions and down in others, with no single
  minimum or maximum.

This is exactly why a loss function's Hessian being PD at a point matters: it
confirms that point is sitting in a genuine bowl (a true local minimum), not
perched on a saddle where gradient descent could still find a downhill
direction.

In [ ]:
def plot_quadratic_form(ax, A_mat, title):
    xs = torch.linspace(-2, 2, 60)
    ys = torch.linspace(-2, 2, 60)
    Xg, Yg = torch.meshgrid(xs, ys, indexing='xy')
    Zg = A_mat[0,0]*Xg**2 + (A_mat[0,1]+A_mat[1,0])*Xg*Yg + A_mat[1,1]*Yg**2
    contourf = ax.contourf(Xg, Yg, Zg, levels=20, cmap='RdBu_r')
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')
    return contourf

pd_matrix = torch.tensor([[2.0, 0.3], [0.3, 1.5]])       # both eigenvalues positive
indefinite_matrix = torch.tensor([[1.0, 0.0], [0.0, -1.0]])  # mixed-sign eigenvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
plot_quadratic_form(axes[0], pd_matrix,
    f"Positive definite: eigenvalues = {torch.linalg.eigvalsh(pd_matrix).tolist()}\n(bowl -- one global min)")
plot_quadratic_form(axes[1], indefinite_matrix,
    f"Indefinite: eigenvalues = {torch.linalg.eigvalsh(indefinite_matrix).tolist()}\n(saddle -- no single min/max)")
plt.tight_layout()
plt.show()

In [ ]:
# Positive semi-definite check via eigenvalues
# Covariance matrices are always PSD
data = torch.randn(100, 4)
cov = torch.cov(data.T)  # 4x4 covariance matrix

eigvals_cov = torch.linalg.eigvalsh(cov)  # eigvalsh -- for symmetric matrices, returns real eigenvalues
print("covariance matrix eigenvalues:", eigvals_cov)
print("all non-negative (PSD)?", torch.all(eigvals_cov >= -1e-6).item())

# A non-PSD example for contrast
not_psd = torch.tensor([[0.0, 1.0], [1.0, 0.0]])
eigvals_not = torch.linalg.eigvalsh(not_psd)
print("\nnon-PSD example eigenvalues:", eigvals_not, "(has a negative eigenvalue)")

### Exercise 6

1. Using autograd, compute the gradient of $f(x) = \|x\|_2^2$ for a random vector
   `x` of dimension 6, and verify it equals $2x$ analytically.
2. Compute the Jacobian of a linear function $f(x) = Ax$ for a random matrix `A`
   (shape 3x4) and input `x` (dimension 4). What do you expect the Jacobian to
   equal, exactly?
3. Build a random symmetric matrix, check if it's PSD via its eigenvalues, and if
   it's not, explain (in a comment) what that means about a loss landscape with
   that matrix as its Hessian at some point.

In [ ]:
# Your turn
torch.manual_seed(8)

# TODO 1: gradient of ||x||_2^2

# TODO 2: Jacobian of a linear function f(x) = Ax

# TODO 3: PSD check on a random symmetric matrix


---
## Capstone — A Tiny Neural Network Built From Raw Tensors

No `nn.Module`, no `nn.Linear` — just tensors, matrix multiplication, broadcasting,
and autograd. This ties together **every** concept from the modules above:

- Module 0: tensors & shapes
- Module 2: matrix multiplication
- Module 3: broadcasting
- Module 4: linear transformations (`y = Wx + b`, done manually — here in the
  row-major layout, `x @ W1 + b1`)
- Module 6: gradients via autograd

We'll train a tiny 2-layer network to solve the classic **XOR problem** — a
dataset that isn't linearly separable, so it genuinely requires a hidden layer
and a non-linearity to solve.

In [ ]:
torch.manual_seed(42)

# XOR dataset
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# Manually initialize weights for a 2 -> 4 -> 1 network.
# NOTE the parentheses: scale FIRST, then ask for gradients. Writing
# `torch.randn(2, 4, requires_grad=True) * 0.5` would make the scaled result a
# non-leaf tensor (the output of a multiply), so autograd would never populate
# its .grad and the update step below would fail.
W1 = (torch.randn(2, 4) * 0.5).requires_grad_(True)
b1 = torch.zeros(4, requires_grad=True)
W2 = (torch.randn(4, 1) * 0.5).requires_grad_(True)
b2 = torch.zeros(1, requires_grad=True)

params = [W1, b1, W2, b2]
lr = 0.5

def forward(x):
    h = torch.relu(x @ W1 + b1)   # linear transform + broadcast bias + nonlinearity
    out = torch.sigmoid(h @ W2 + b2)
    return out

losses = []
for epoch in range(2000):
    y_pred = forward(X_xor)
    loss = torch.mean((y_pred - y_xor) ** 2)   # MSE loss

    # zero gradients, backprop, manual gradient step
    for p in params:
        if p.grad is not None:
            p.grad.zero_()
    loss.backward()
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    losses.append(loss.item())
    if epoch % 400 == 0:
        print(f"epoch {epoch:4d} | loss {loss.item():.4f}")

print("\nFinal predictions:")
print(forward(X_xor).detach())
print("Targets:")
print(y_xor)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.title('Training loss -- tiny raw-tensor neural net on XOR')
plt.show()

### Capstone reflection

Every operation above is something covered in the modules:
- `x @ W1 + b1` — Module 4's affine transform $\mathbf{y} = W\mathbf{x} + \mathbf{b}$,
  written row-major because the batch dimension comes first (`W1` is stored as
  `(in, out)` here, so no transpose is needed), with broadcasting (Module 3)
  adding the bias across the batch
- `torch.relu`, `torch.sigmoid` — nonlinearities applied element-wise
- `loss.backward()` — autograd computing gradients through the whole chain
  (Module 6), including implicitly through every matrix multiplication (Module 2)
- `p -= lr * p.grad` — manual gradient descent, the same update `nn.Module` +
  `torch.optim` do for you automatically

**Try extending this yourself:**
- Swap `torch.optim.Adam` in for the manual update loop
- Add a second hidden layer
- Replace the manual weight init with `torch.nn.init.orthogonal_` and see if
  training behaves differently (callback to Module 2's orthogonal matrices!)

---

**End of tutorial.** You've now covered tensors, vector/matrix operations, norms,
trace, determinant, orthogonality, broadcasting, batched ops, linear
transformations, rank, eigenvalues, SVD, PCA, low-rank approximation, gradients,
Jacobians, and positive definiteness — all through runnable PyTorch code.

---
## Solutions

The solutions for all exercises are collected here so you can try each exercise before checking the answer.


### Solution 0


In [ ]:
print(torch.zeros(3, 5).shape)                       # torch.Size([3, 5])
print(torch.ones(2, 3, 4).shape)                      # torch.Size([2, 3, 4])
print(torch.tensor([1, 2, 3]).unsqueeze(0).shape)     # torch.Size([1, 3])  -- adds a dim
print(torch.tensor([[1, 2, 3]]).squeeze(0).shape)     # torch.Size([3])     -- removes a dim

### Solution 1


In [ ]:
torch.manual_seed(1)
v1 = torch.randn(5)
v2 = torch.randn(5)

cos_before = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
print("cosine similarity (original):", cos_before.item())

v1_scaled = v1 * 10
cos_after = F.cosine_similarity(v1_scaled.unsqueeze(0), v2.unsqueeze(0))
print("cosine similarity (v1 scaled 10x):", cos_after.item())
print("-> unchanged! cosine similarity only depends on the ANGLE between vectors, not magnitude.")

outer_v = torch.outer(v1, v2)
print("outer product shape:", outer_v.shape)  # (5, 5)

### Solution 2


In [ ]:
torch.manual_seed(2)

# 1. Frobenius norm via trace
M = torch.randn(3, 3)
lhs = torch.norm(M, 'fro')
rhs = torch.sqrt(torch.trace(M.T @ M))
print("Frobenius norm:", lhs.item(), "| sqrt(trace(M.T@M)):", rhs.item())

# 2. Cyclic trace property
A2, B2, C2 = torch.randn(3, 3), torch.randn(3, 3), torch.randn(3, 3)
t1 = torch.trace(A2 @ B2 @ C2)
t2 = torch.trace(C2 @ A2 @ B2)
t3 = torch.trace(B2 @ C2 @ A2)
print("\ntrace(ABC):", t1.item())
print("trace(CAB):", t2.item())
print("trace(BCA):", t3.item())

# 3. Orthogonality check for a 3D rotation around the y-axis
angle = torch.tensor(1.234)
Q2 = torch.tensor([
    [ torch.cos(angle).item(), 0.0, torch.sin(angle).item()],
    [0.0,                     1.0, 0.0],
    [-torch.sin(angle).item(), 0.0, torch.cos(angle).item()],
])
print("\nQ.T @ Q:\n", Q2.T @ Q2, "\n(should be very close to the identity matrix)")

### Solution 3


In [ ]:
torch.manual_seed(3)

# 1. For-loop vs. vectorized row-wise mean
import time
N3, C3 = 5000, 256
X3 = torch.randn(N3, C3)

start = time.time()
means_loop = torch.zeros(N3)
for i in range(N3):
    means_loop[i] = torch.mean(X3[i])
loop_t = time.time() - start

start = time.time()
means_vec = torch.mean(X3, dim=1)
vec_t = time.time() - start

print(f"loop time: {loop_t:.4f}s | vectorized time: {vec_t:.4f}s | speedup: {loop_t/vec_t:.1f}x")
print("match:", torch.allclose(means_loop, means_vec, atol=1e-5))

# 2. Flatten batch of images
imgs = torch.randn(32, 1, 64, 64)
flat = imgs.reshape(32, -1)
print("\nflattened shape:", flat.shape)

# 3. Batched dot product
a3 = torch.randn(6, 10)
b3 = torch.randn(6, 10)
dot_einsum = torch.einsum('bd,bd->b', a3, b3)
dot_manual = torch.stack([torch.dot(a3[i], b3[i]) for i in range(a3.shape[0])])
print("\neinsum result:", dot_einsum)
print("manual loop result:", dot_manual)
print("match:", torch.allclose(dot_einsum, dot_manual, atol=1e-5))

# 4. Invalid reshape (commented so the notebook doesn't stop -- uncomment to see the error)
try:
    torch.arange(10).reshape(3, 4)  # 10 elements can't fill a 3x4=12 tensor
except RuntimeError as e:
    print("\nExpected error:", e)


### Solution 4


In [ ]:
torch.manual_seed(5)

x4 = torch.randn(3, 4)     # a batch of 3 samples, stored as rows (PyTorch's layout)

# 1-3. Single layer, textbook form y = Wx + b, one sample at a time
layer1 = nn.Linear(4, 2)
x_one = x4[0]                                   # (4,)
manual_one = layer1.weight @ x_one + layer1.bias
print("single sample match:", torch.allclose(layer1(x_one), manual_one, atol=1e-6))

# ...and the same layer over the whole batch, samples as columns: Y = W X + b
manual_batch = (layer1.weight @ x4.T + layer1.bias.unsqueeze(1)).T
print("whole batch match:  ", torch.allclose(layer1(x4), manual_batch, atol=1e-6))

# 4. Two-layer MLP, keeping the samples in columns the whole way through
mlp = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
official_out = mlp(x4)
l1, l2 = mlp[0], mlp[2]

H = torch.relu(l1.weight @ x4.T + l1.bias.unsqueeze(1))   # (8, batch)
Out = l2.weight @ H + l2.bias.unsqueeze(1)                 # (2, batch)
print("two-layer MLP match:", torch.allclose(official_out, Out.T, atol=1e-6))

# For contrast: the row-major form, which is what nn.Linear runs internally.
# Same numbers, fewer transposes -- that convenience is the only reason PyTorch
# prefers it over the y = Wx + b you would write on paper.
h_rows = torch.relu(x4 @ l1.weight.T + l1.bias)
out_rows = h_rows @ l2.weight.T + l2.bias
print("row-major form match:", torch.allclose(official_out, out_rows, atol=1e-6))

### Solution 5


In [ ]:
torch.manual_seed(7)

# 1. Rank of a constructed low-rank matrix
low_rank_6x6 = torch.randn(6, 2) @ torch.randn(2, 6)
print("rank:", torch.linalg.matrix_rank(low_rank_6x6).item(), "(expected: 2, since it's a product through a 2-dim bottleneck)")

# 2. Variance explained by top 5 components (reusing U, S, Vt from the digits cell above)
var_explained_5 = torch.sum(S[:5] ** 2) / torch.sum(S ** 2)
print("variance explained by top 5 components:", f"{var_explained_5.item():.2%}")

# 3. Reconstruction with top 10 components
k = 10
X_approx = U[:, :k] @ torch.diag(S[:k]) @ Vt[:k, :]
recon_error = torch.norm(X_centered - X_approx, 'fro').item()
print(f"reconstruction error with top {k} components: {recon_error:.3f}")

### Solution 6


In [ ]:
torch.manual_seed(8)

# 1. Gradient of ||x||_2^2
x6 = torch.randn(6, requires_grad=True)
f6 = torch.norm(x6, p=2) ** 2
f6.backward()
print("autograd grad:", x6.grad)
print("analytical 2x:", 2 * x6.detach())
print("match:", torch.allclose(x6.grad, 2 * x6.detach(), atol=1e-4))

# 2. Jacobian of a linear function f(x) = Ax
A6 = torch.randn(3, 4)
def linear_fn(x):
    return A6 @ x
x_lin = torch.randn(4)
J_lin = torch.autograd.functional.jacobian(linear_fn, x_lin)
print("\nJacobian of Ax:\n", J_lin)
print("matches A exactly:", torch.allclose(J_lin, A6, atol=1e-5))
# Makes sense: for a linear map f(x) = Ax, the Jacobian IS A, everywhere.

# 3. PSD check
M6 = torch.randn(4, 4)
sym_M6 = (M6 + M6.T) / 2  # force symmetry
eigvals6 = torch.linalg.eigvalsh(sym_M6)
print("\neigenvalues:", eigvals6)
is_psd = torch.all(eigvals6 >= -1e-6).item()
print("is PSD:", is_psd)
# If NOT PSD (has a negative eigenvalue): if this matrix were a loss's Hessian at
# some point, that point would be a saddle point rather than a local minimum --
# the loss decreases in at least one direction, so gradient descent isn't "done" there.